# Education level per municipality

This notebook maps the share of highly and low educated residents in all Dutch municipalities, and saves the result for the analysis.

**Definitions** (persons aged 15 to 75):
- **Highly educated:** hbo or wo as highest completed level
- **Low educated:** basisonderwijs, vmbo or mbo-1 as highest completed level
- Shares are calculated as a percentage of all persons aged 15 to 75 with a known education level (low + middle + high).

**Required files:**
- `data/raw/kwb2024_per_gemeente.csv`: education level per municipality, extracted from CBS Kerncijfers wijken en buurten 2024 (85984NED)
- `data/raw/gemeenten_2024.gpkg`: municipal boundaries from CBS/PDOK, 2024 municipal division
- `data/processed/chargingpoints_households_income.csv`: created by notebook 1, only used to check that the names match

**Output:**
- `data/processed/education_per_municipality.csv`
- `figures/map_education_per_municipality.png`

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

## 1. Load the data

The file contains the number of persons aged 15 to 75 per education level (`opl_laag`, `opl_middel`, `opl_hoog`). We keep only the municipality code, the name and these three columns.

In [ ]:
kwb = pd.read_csv("../data/raw/kwb2024_per_gemeente.csv")

education = kwb[["gemeentecode", "gemeente", "opl_laag", "opl_middel", "opl_hoog"]].rename(columns={
    "gemeentecode": "municipality_code",
    "gemeente": "municipality",
    "opl_laag": "low_educated",
    "opl_middel": "middle_educated",
    "opl_hoog": "highly_educated",
})

print("Number of municipalities:", len(education))
display(education.head())

## 2. Calculate the shares

The shares are only calculated when all three education levels are known. For Rozendaal, CBS does not publish the number of low educated persons (too few residents), so its shares are left empty instead of being calculated from an incomplete total.

In [ ]:
total_known = education[["low_educated", "middle_educated", "highly_educated"]].sum(axis=1, min_count=3)

education["highly_educated_pct"] = (education["highly_educated"] / total_known * 100).round(2)
education["low_educated_pct"] = (education["low_educated"] / total_known * 100).round(2)

print("Municipalities without a share:", education.loc[education["highly_educated_pct"].isna(), "municipality"].tolist())
display(education[["highly_educated_pct", "low_educated_pct"]].describe().round(1))

## 3. Join the data to the map

Both files contain the CBS municipality code (for example `GM0014`), so the data can be joined on the code instead of the name.

In [ ]:
municipalities = gpd.read_file("../data/raw/gemeenten_2024.gpkg", layer="gemeente_gegeneraliseerd")

map_gdf = municipalities.merge(education, left_on="statcode", right_on="municipality_code", how="left")

# Check: should print 0 and 1 (Rozendaal)
print("Municipalities not found in the data:", map_gdf["municipality_code"].isna().sum())
print("Municipalities without a share:", map_gdf["highly_educated_pct"].isna().sum())

## 4. Create the maps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 10))

map_gdf.plot(
    column="highly_educated_pct",
    cmap="Oranges",
    linewidth=0.3,
    edgecolor="white",
    legend=True,
    legend_kwds={"label": "Highly educated (% of residents aged 15-75)", "shrink": 0.6},
    ax=axes[0],
    missing_kwds={"color": "lightgrey", "label": "No data"},
)
axes[0].set_title("Share of highly educated residents\n(hbo/wo), 2024", fontsize=14)
axes[0].axis("off")

map_gdf.plot(
    column="low_educated_pct",
    cmap="PuBu",
    linewidth=0.3,
    edgecolor="white",
    legend=True,
    legend_kwds={"label": "Low educated (% of residents aged 15-75)", "shrink": 0.6},
    ax=axes[1],
    missing_kwds={"color": "lightgrey", "label": "No data"},
)
axes[1].set_title("Share of low educated residents\n(basisonderwijs, vmbo, mbo-1), 2024", fontsize=14)
axes[1].axis("off")

plt.tight_layout()
os.makedirs("../figures", exist_ok=True)
plt.savefig("../figures/map_education_per_municipality.png", dpi=200)
plt.show()

## 5. Save for the analysis

We use the same municipality names as the charging point file (the same mapping as in notebook 2), so all processed files can be merged on `municipality`.

In [ ]:
name_mapping = {
    "'s-Gravenhage": "Den Haag",
    "Beek (L.)": "Beek",
    "Hengelo (O.)": "Hengelo",
    "Laren (NH.)": "Laren",
    "Middelburg (Z.)": "Middelburg",
    "Rijswijk (ZH.)": "Rijswijk",
    "Stein (L.)": "Stein",
}
education_save = education[["municipality", "highly_educated_pct", "low_educated_pct"]].copy()
education_save["municipality"] = education_save["municipality"].str.strip().replace(name_mapping)

# Check: does every municipality match the charging point data?
charging = pd.read_csv("../data/processed/chargingpoints_households_income.csv")
print("Only in the education data:", set(education_save["municipality"]) - set(charging["municipality"]))
print("Only in the charging data:", set(charging["municipality"]) - set(education_save["municipality"]))

education_save.to_csv("../data/processed/education_per_municipality.csv", index=False)
print("Saved:", len(education_save), "municipalities")